# 01 · Ingesta y caracterización

Carga las fuentes a formato *tidy* usando los módulos de `src/`, verifica el dato,
caracteriza el resultado y guarda salidas intermedias en `data/interim/`.

Fuentes en este notebook: **SNIIV** (cubo Infonavit, municipio × segmento) e
**Inmuebles24** (listados, oferta). Las trampas del dato están documentadas en
[`docs/decisiones.md`](../docs/decisiones.md).

## Configuración

In [1]:
from pathlib import Path
import sys

import pandas as pd

# Raíz del repo 
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
sys.path.append(str(REPO / "src"))

from ingesta_sniiv import cargar_sniiv
from ingesta_inmuebles import cargar_inmuebles

RAW = REPO / "data" / "raw"
INTERIM = REPO / "data" / "interim"
INTERIM.mkdir(exist_ok=True)

## 1. SNIIV — cubo Infonavit

Se localizan los dos extractos por patrón de nombre (robusto a la fecha del archivo).

In [2]:
ruta_acciones = next(RAW.glob("infonavit_acciones-segmento_merida_*.xls"))
ruta_monto = next(RAW.glob("infonavit_monto-segmento_merida_*.xls"))

sniiv = cargar_sniiv(ruta_acciones, ruta_monto)
sniiv.head()

,anio,segmento,acciones,monto
0,2015,Económica,306,5.464735e+07
1,2015,Media,697,2.812566e+08
2,2015,No disponible,3536,0.000000e+00
3,2015,Popular,3426,8.865023e+08
4,2015,Residencial,74,3.108829e+07


### Forma y tipos

In [3]:
print("Forma:", sniiv.shape)  # (77, 4) = 11 años × 7 segmentos
sniiv.dtypes

Forma: (77, 4)


anio          int64
segmento        str
acciones      Int64
monto       float64
dtype: object

### Totales y validación

Se valida el pipeline contra los totales conocidos del cubo (Mérida 2015–2025):
**69,448** acciones y **$24,493,507,058.80** de monto.

In [4]:
total_acciones = int(sniiv["acciones"].sum())
total_monto = float(sniiv["monto"].sum())
print(f"Total acciones: {total_acciones:,}")
print(f"Total monto:    ${total_monto:,.2f}")

assert total_acciones == 69_448, total_acciones
assert round(total_monto, 2) == 24_493_507_058.80, total_monto
print("Chequeos de cordura: OK")

Total acciones: 69,448
Total monto:    $24,493,507,058.80
Chequeos de cordura: OK


### Distribución por segmento

`monto_prom_credito` = monto / acciones es el **crédito promedio**, no el valor de
la vivienda (ver `docs/decisiones.md`).

In [5]:
resumen = (
    sniiv.groupby("segmento")
    .agg(acciones=("acciones", "sum"), monto=("monto", "sum"))
    .assign(
        part_acciones=lambda x: (x["acciones"] / x["acciones"].sum() * 100).round(1),
        monto_prom_credito=lambda x: (x["monto"] / x["acciones"]).round(0),
    )
    .sort_values("acciones", ascending=False)
)
resumen

,acciones,monto,part_acciones,monto_prom_credito
segmento,,,,
No disponible,19808,8.435948e+06,28.5,426.0
Popular,19120,6.382306e+09,27.5,333803.0
Tradicional,17521,8.381496e+09,25.2,478369.0
Media,10493,8.166900e+09,15.1,778319.0
Residencial,1432,1.314475e+09,2.1,917930.0
Económica,917,1.805680e+08,1.3,196912.0
Residencial plus,157,5.932564e+07,0.2,377870.0


### "No disponible"

Casi un tercio de los créditos, con monto ≈ 0. Se reporta como dato, pero se
excluye del análisis de valor.

In [6]:
mask = sniiv["segmento"] == "No disponible"
acc_nd = int(sniiv.loc[mask, "acciones"].sum())
print(f"No disponible — acciones: {acc_nd:,} "
      f"({acc_nd / total_acciones * 100:.1f}% del total)")
print(f"No disponible — monto:    ${sniiv.loc[mask, 'monto'].sum():,.2f}")

No disponible — acciones: 19,808 (28.5% del total)
No disponible — monto:    $8,435,947.99


### Salida intermedia

In [7]:
salida_sniiv = INTERIM / "sniiv_merida_tidy.parquet"
sniiv.to_parquet(salida_sniiv, index=False)
print("Guardado:", salida_sniiv.relative_to(REPO))

Guardado: data/interim/sniiv_merida_tidy.parquet


## 2. Inmuebles24 — listados (oferta)

Dataset propio del proyecto previo (`propiedades.csv`). `precio` es **precio de
lista**, no de cierre.

In [8]:
inm = cargar_inmuebles(RAW / "propiedades.csv")
inm.head()

,fecha_registro,url,url_archivo,operación,tipo_inmueble,colonia,precio,m2_construccion,m2_terreno,recamaras,banos,estacionamientos,antigüedad,es_preventa,notas
0,2026-05-29,https://www.inmuebles24.com/propiedades/clasif...,NaN,venta,casa,altabrisa,2390000,100,160,2,1,2,20,False,Casa en fraccionamiento Vista Alegre Norte. Ca...
1,2026-05-29,https://www.inmuebles24.com/propiedades/clasif...,NaN,venta,departamento,altabrisa,2285000,46,46,1,1,1,<NA>,True,"Preventa, múltiples unidades, precio cabecera ..."
2,2026-05-29,https://www.inmuebles24.com/propiedades/clasif...,NaN,venta,departamento,altabrisa,6200000,160,160,2,3,2,<NA>,False,"Country Towers Altabrisa, anuncio no especific..."
3,2026-05-29,https://www.inmuebles24.com/propiedades/clasif...,NaN,venta,casa,altabrisa,7500000,292,326,4,3,2,10,False,"Sin cuota de mantenimiento, piscina con cascad..."
4,2026-05-29,https://www.inmuebles24.com/propiedades/clasif...,NaN,venta,casa,altabrisa,27000000,800,890,4,4,2,6,False,"Casa en condominio, piscina con baño completo,..."


### Forma, tipos y faltantes

In [9]:
print("Forma:", inm.shape)
inm.dtypes

Forma: (60, 15)


fecha_registro      datetime64[us]
url                            str
url_archivo                float64
operación                      str
tipo_inmueble                  str
colonia                        str
precio                       int64
m2_construccion              int64
m2_terreno                   int64
recamaras                    int64
banos                        int64
estacionamientos             Int64
antigüedad                   Int64
es_preventa                boolean
notas                          str
dtype: object

In [10]:
faltantes = inm.isna().sum()
faltantes[faltantes > 0].rename("faltantes")

url_archivo         60
estacionamientos     1
antigüedad          39
notas                1
Name: faltantes, dtype: int64

### Caracterización

Mezcla por tipo, preventa, colonias y rango de precios de lista.

In [11]:
print("Tipo de inmueble:", inm["tipo_inmueble"].value_counts().to_dict())
print("Preventa:        ", inm["es_preventa"].value_counts().to_dict())
print("Colonias:        ", inm["colonia"].nunique(), "→", sorted(inm["colonia"].unique()))
print(f"Precio de lista:  ${inm['precio'].min():,} – ${inm['precio'].max():,} "
      f"(mediana ${inm['precio'].median():,.0f})")

Tipo de inmueble: {'casa': 44, 'departamento': 16}
Preventa:         {np.False_: 50, np.True_: 10}
Colonias:         6 → ['altabrisa', 'chuburna de hidalgo', 'francisco de montejo', 'garcia gineres', 'santa gertrudis copo', 'yucatan country club']
Precio de lista:  $1,790,000 – $41,000,000 (mediana $4,942,500)


### Salida intermedia

In [12]:
salida_inm = INTERIM / "inmuebles_tidy.parquet"
inm.to_parquet(salida_inm, index=False)
print("Guardado:", salida_inm.relative_to(REPO))

Guardado: data/interim/inmuebles_tidy.parquet
